In [1]:
import os, sys
import random
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data_dir= "/ais/bulbasaur/twkillian/AHE_Sepsis_Data/rectilinear_processed/"
data_dir2 = "/ais/bulbasaur/twkillian/AHE_Sepsis_Data/"

In [3]:
encoded_train_data = np.load(data_dir+"encoded_{}.npz".format("train"))
encoded_val_data = np.load(data_dir+"encoded_{}.npz".format("validation"))

In [5]:
list(encoded_train_data.keys())

['states', 'actions', 'rewards', 'lengths', 'stay_ids']

In [6]:
encoded_states_t = encoded_train_data['states']
encoded_states_v = encoded_val_data['states']

In [7]:
encoded_states_t.shape, encoded_states_v.shape

((5805, 73, 53), (387, 73, 53))

In [8]:
sids_t = encoded_train_data['stay_ids']
len(sids_t)

5805

### Gather the NS/S ids from the training set and downsample to {10%, 25%, 50%, 75%}, then extract the data corresponding to those IDs to form the new training sets... 

In [10]:
data = pd.read_csv(os.path.join(data_dir2, "final_trajs_noFill_RAW.csv"), compression="gzip")
data.head()

,m:traj,m:timestep,m:bloc,m:stay_id,m:time_from_sepsis_h,o:gender,o:ventilation,o:admission_age,o:height,o:weight,...,o:spo2,o:bun,o:creatinine,o:inr,o:bilirubin,o:alt,o:ast,o:UO,a:action,r:reward
0,1,0,-8,30004823.0,-8.000000,1,0.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,250.0,0.0,1.0
1,1,1,-7,30004823.0,-7.410000,1,1.0,53.0,175.0,91.0,...,97.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
2,1,2,-6,30004823.0,-6.491667,1,1.0,53.0,175.0,91.0,...,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
3,1,3,-5,30004823.0,-5.491667,1,1.0,53.0,175.0,91.0,...,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
4,1,4,-4,30004823.0,-4.433333,1,1.0,53.0,175.0,91.0,...,95.0,29.0,0.6,2.3,NaN,NaN,NaN,NaN,15.0,1.0


In [11]:
red_data = data.groupby('m:stay_id').agg('first').reset_index()[['m:stay_id', 'm:time_from_sepsis_h', 'r:reward']]
red_data.head()

,m:stay_id,m:time_from_sepsis_h,r:reward
0,30004823.0,-8.000000,1.0
1,30005160.0,8.773333,1.0
2,30010785.0,2.688889,1.0
3,30011071.0,5.608333,1.0
4,30011624.0,-3.500000,1.0


In [13]:
red_data_t = red_data[red_data['m:stay_id'].isin(sids_t)]
red_data_t.shape

(5805, 3)

In [15]:
ns_ids = red_data_t[red_data_t['r:reward']<0]['m:stay_id'].values
s_ids = red_data_t[red_data_t['r:reward']>0]['m:stay_id'].values

In [16]:
ns_ids[:5], s_ids[:5]

(array([30052347., 30056474., 30069144., 30082050., 30097127.]),
 array([30005160., 30010785., 30011071., 30011624., 30018144.]))

In [17]:
len(ns_ids), len(s_ids)

(713, 5092)

In [21]:
p10_ns_ids, p10_s_ids = np.random.choice(ns_ids, size=round(.1*len(ns_ids)), replace=False), np.random.choice(s_ids, size=round(.1*len(s_ids)), replace=False)
p25_ns_ids, p25_s_ids = np.random.choice(ns_ids, size=round(.25*len(ns_ids)), replace=False), np.random.choice(s_ids, size=round(.25*len(s_ids)), replace=False)
p50_ns_ids, p50_s_ids = np.random.choice(ns_ids, size=round(.5*len(ns_ids)), replace=False), np.random.choice(s_ids, size=round(.5*len(s_ids)), replace=False)
p75_ns_ids, p75_s_ids = np.random.choice(ns_ids, size=round(.75*len(ns_ids)), replace=False), np.random.choice(s_ids, size=round(.75*len(s_ids)), replace=False)

In [23]:
len(p10_ns_ids), len(p10_s_ids)

(71, 509)

In [41]:
p10_ids_t = np.concatenate((p10_ns_ids, p10_s_ids))
p25_ids_t = np.concatenate((p25_ns_ids, p25_s_ids))
p50_ids_t = np.concatenate((p50_ns_ids, p50_s_ids))
p75_ids_t = np.concatenate((p75_ns_ids, p75_s_ids))

In [42]:
p10_idx = np.isin(sids_t, p10_ids_t)
p25_idx = np.isin(sids_t, p25_ids_t)
p50_idx = np.isin(sids_t, p50_ids_t)
p75_idx = np.isin(sids_t, p75_ids_t)

In [43]:
# Create npz object and save the training data 
npz_p10_dict = {}

npz_p10_dict['states'] = encoded_train_data['states'][p10_idx]
npz_p10_dict['actions'] = encoded_train_data['actions'][p10_idx]
npz_p10_dict['rewards'] = encoded_train_data['rewards'][p10_idx]
npz_p10_dict['lengths'] = encoded_train_data['lengths'][p10_idx]
npz_p10_dict['stay_ids'] = encoded_train_data['stay_ids'][p10_idx]


np.savez("/ais/bulbasaur/twkillian/AHE_Sepsis_Data/rectilinear_processed/encoded_train_p10.npz", **npz_p10_dict)

In [44]:
# Create npz object and save the training data 
npz_p25_dict = {}

npz_p25_dict['states'] = encoded_train_data['states'][p25_idx]
npz_p25_dict['actions'] = encoded_train_data['actions'][p25_idx]
npz_p25_dict['rewards'] = encoded_train_data['rewards'][p25_idx]
npz_p25_dict['lengths'] = encoded_train_data['lengths'][p25_idx]
npz_p25_dict['stay_ids'] = encoded_train_data['stay_ids'][p25_idx]


np.savez("/ais/bulbasaur/twkillian/AHE_Sepsis_Data/rectilinear_processed/encoded_train_p25.npz", **npz_p25_dict)

In [45]:
# Create npz object and save the training data 
npz_p50_dict = {}

npz_p50_dict['states'] = encoded_train_data['states'][p50_idx]
npz_p50_dict['actions'] = encoded_train_data['actions'][p50_idx]
npz_p50_dict['rewards'] = encoded_train_data['rewards'][p50_idx]
npz_p50_dict['lengths'] = encoded_train_data['lengths'][p50_idx]
npz_p50_dict['stay_ids'] = encoded_train_data['stay_ids'][p50_idx]


np.savez("/ais/bulbasaur/twkillian/AHE_Sepsis_Data/rectilinear_processed/encoded_train_p50.npz", **npz_p50_dict)

In [46]:
# Create npz object and save the training data 
npz_p75_dict = {}

npz_p75_dict['states'] = encoded_train_data['states'][p75_idx]
npz_p75_dict['actions'] = encoded_train_data['actions'][p75_idx]
npz_p75_dict['rewards'] = encoded_train_data['rewards'][p75_idx]
npz_p75_dict['lengths'] = encoded_train_data['lengths'][p75_idx]
npz_p75_dict['stay_ids'] = encoded_train_data['stay_ids'][p75_idx]


np.savez("/ais/bulbasaur/twkillian/AHE_Sepsis_Data/rectilinear_processed/encoded_train_p75.npz", **npz_p75_dict)

In [10]:
test = np.vstack((encoded_states_t, encoded_states_v))

In [11]:
test.shape

(6192, 73, 53)

In [35]:
class RLDataLoader(object):
    def __init__(self, data_dir, rng, minibatch_size, pos_samples_in_minibatch=0, neg_samples_in_minibatch=0, drop_smaller_than_minibatch=False, dataset='train', device='cpu'):
        '''
        Custom DataLoader that ensures that transitons from both pos/neg trajectories are sampled in a balanced way.

        Args:
            data_dir (str): pointer to the location of the .npz files where the encoded data is stored
            rng (random): the random number generator that was established for reproducibility
            minibatch_size (int): the size of minibatches that will be sampled using this dataloader
            drop_smaller_than_minibatch (int?): TBD?
            dataset (str): default='train', options = ['train', 'validation', 'test', 'overlap']

        Use: first call make_transition_train_data() once, then call reset() before each epoch of getting all minibatches
        '''
        self.rng = rng
        self.minibatch_size = minibatch_size
        self.drop_smaller_than_minibatch = drop_smaller_than_minibatch
        self.ps = pos_samples_in_minibatch  # The number of positive terminal transtions placed in each minibatch
        self.ns = neg_samples_in_minibatch  # The number of negative terminal transitons place in each minibatch
        self.device = device
        # Load the data (encoded data is stored as .npz with files ['states', 'actions', 'rewards', 'lengths'])
        if dataset == 'train_val': # Combine the training and validation dataset (final training of best performing model...)
            tr_data = np.load(data_dir+"encoded_train.npz")
            v_data = np.load(data_dir+"encoded_validation.npz")

            # Combine the encoded states
            self.encoded_data = np.vstack((tr_data['states'], v_data['states']))
            # Combine the actions, ensure that they're integers and remove unncessary dims
            self.encoded_actions = np.vstack(
                (
                    tr_data['actions'].astype(int).squeeze(),
                    v_data['actions'].astype(int).squeeze()
                )
            )
            # Combine the rewards, remove unecessary dims
            self.encoded_rewards = np.vstack((tr_data['rewards'].squeeze(), v_data['rewards'].squeeze()))
            # Combine the recorded lengths of each trajectory, change to integers (for indexing purposes)
            self.encoded_lengths = np.vstack((tr_data['lengths'].astype(int), v_data['lengths'].astype(int))).squeeze()
        else:
            encoded_data_file = np.load(data_dir+"encoded_{}.npz".format(dataset))
            self.encoded_data = encoded_data_file['states']
            # Ensure that the actions are recorded as integers since they'll be used to index the Q-value returns
            self.encoded_actions = encoded_data_file['actions'].astype(int).squeeze()  # Removing unneeded extra final dimension... If (N, T, 1) -> (N, T)... If (N,T,num_actions) (one-hot encoded), then nothing changes
            self.encoded_rewards = encoded_data_file['rewards'].squeeze()  # Removing unneeded extra final dimension... (N, T, 1) -> (N,T)
            # Change the recorded lengths of each trajectory since this quantity is used to index the other arrays
            self.encoded_lengths = encoded_data_file['lengths'].astype(int).squeeze() # Removing unneeded extra final dimension... (N, 1) -> (N,)

        # Extract the dimension of the encoded states to return for building the Q-networks
        self.data_dim = self.encoded_data.shape[-1]

        # Set up data structures and indexing mechanisms that will be used to sample minibatches
        self.transition_data = {}
        self.transition_indices_pos_last = []
        self.transition_indices_neg_last = []
        self.transition_indices = None
        self.transition_data_size = None
        self.transitions_head = None
        self.transitions_head_pos = None
        self.transitions_head_neg = None
        self.epoch_finished = True  # to enforce reset() before use
        self.epoch_pos_finished = True
        self.epoch_neg_finished = True
        self.num_minibatches_epoch = None
    
    def reset(self, shuffle):
        if shuffle:
            self.rng.shuffle(self.transition_indices)
            self.rng.shuffle(self.transition_indices_pos_last)
            self.rng.shuffle(self.transition_indices_neg_last)
        # Reset the index/counters for traversing the stored data arrays when "sampling" minibatches
        self.transitions_head = 0
        self.transitions_head_pos = 0
        self.transitions_head_neg = 0
        # Reset the flag indicators for when we've exhausted the available data
        self.epoch_finished = False
        self.epoch_pos_finished = False
        self.epoch_neg_finished = False

    def make_transition_data(self, release=False):
        print("DataLoader: making transitions (s,a,r,s') from encoded data structures")
        self.transition_data['s'] = {}
        self.transition_data['actions'] = {}
        self.transition_data['rewards'] = {}
        self.transition_data['next_s'] = {}
        self.transition_data['terminals'] = {}
        indices_pos = []
        indices_neg = []
        counter = 0
        
        for traj in range(self.encoded_data.shape[0]):
            # Check whether terminal reward for this trajectory is positive (patient survived)
            traj_is_positive = self.encoded_rewards[traj, self.encoded_lengths[traj]-1] > 0
            for t in range(self.encoded_lengths[traj] - 1):
                self.transition_data['s'][counter] = self.encoded_data[traj,t, :]
                self.transition_data['next_s'][counter] = self.encoded_data[traj, t+1, :]
                self.transition_data['actions'][counter] = self.encoded_actions[traj, t]
                self.transition_data['rewards'][counter] = self.encoded_rewards[traj, t]
                self.transition_data['terminals'][counter] = 0
                if traj_is_positive: 
                    indices_pos.append(counter)
                else:
                    indices_neg.append(counter)
                counter += 1
            # For the last transition in the trajectory
            tlast = self.encoded_lengths[traj] - 1 # Get the index of the terminal state
            self.transition_data['s'][counter] = self.encoded_data[traj, tlast, :]
            self.transition_data['next_s'][counter] = np.zeros_like(self.encoded_data[traj, tlast, :])
            self.transition_data['actions'][counter] = self.encoded_actions[traj, tlast]
            self.transition_data['rewards'][counter] = self.encoded_rewards[traj, tlast]
            self.transition_data['terminals'][counter] = 1
            if traj_is_positive:
                self.transition_indices_pos_last.append(counter)
            else:
                self.transition_indices_neg_last.append(counter)
            counter += 1
        self.transition_data_size = counter
        self.transition_indices = np.arange(self.transition_data_size)
        if release:
            del self.encoded_data, self.encoded_actions, self.encoded_rewards
            self.encoded_data, self.encoded_actions, self.encoded_rewards = None, None, None
            gc.collect()

        # Compute the number of minibatches that will be drawn per epoch
        self.num_minibatches_epoch = int(np.floor(self.transition_data_size / self.minibatch_size)) + int(1 - self.drop_smaller_than_minibatch)
    
    def get_next_minibatch(self):
        if self.epoch_finished == True:
            print('Epoch finished, please call reset() method before next call to get_next_minibatch()')
            return None
        # Getting data from dictionaries
        offset = self.ns + self.ps  # The number of samples we'll need to replace in the list of indices below
        minibatch_main_index_list = list(self.transition_indices[self.transitions_head:self.transitions_head + self.minibatch_size - offset])
        # Throughout an epoch, we cycle through the terminal transitions and include them specially in the sampled minibatch as desired
        # self.ps is an integer number of positive terminal states added to the minibatch
        # self.ns is an integer number of negative terminal states added to the minibatch
        minibatch_pos_last_index_list = self.transition_indices_pos_last[self.transitions_head_pos:self.transitions_head_pos + self.ps]
        minibatch_neg_last_index_list = self.transition_indices_neg_last[self.transitions_head_neg:self.transitions_head_neg + self.ns]
        # Increment the starting index for the next terminal state resampling
        self.transitions_head_pos += self.ps
        self.transitions_head_neg += self.ns
        # Combine the index lists to construct our minibatch from using operator.itemgetter
        minibatch_index_list = minibatch_main_index_list + minibatch_pos_last_index_list + minibatch_neg_last_index_list
        get_from_dict = operator.itemgetter(*minibatch_index_list)
        
        # Extract the transition data to form the minibatch
        # operator.itemgetter just appends the extracted subarrays in a tuple, wrap in np.float32 to get ndarrays
        # We also cast these elements to torch Tensors and move to the computation device so they're ready for processing right away
        s_minibatch = torch.from_numpy(np.float32(get_from_dict(self.transition_data['s']))).to(self.device)
        actions_minibatch = torch.from_numpy(np.float32(get_from_dict(self.transition_data['actions']))).to(self.device, dtype=torch.int64)
        rewards_minibatch = torch.from_numpy(np.float32(get_from_dict(self.transition_data['rewards']))).to(self.device)
        next_s_minibatch = torch.from_numpy(np.float32(get_from_dict(self.transition_data['next_s']))).to(self.device)
        terminals_minibatch = torch.from_numpy(np.float32(get_from_dict(self.transition_data['terminals']))).to(self.device)
        
        # Updating current data head to start the next minibatch at the end of the current one
        self.transitions_head += self.minibatch_size
        self.epoch_finished = self.transitions_head + self.drop_smaller_than_minibatch*self.minibatch_size >= self.transition_data_size
        self.transitions_head_pos = self.transitions_head_pos % len(self.transition_indices_pos_last)
        self.transitions_head_neg = self.transitions_head_neg % len(self.transition_indices_neg_last)

        return s_minibatch, actions_minibatch, rewards_minibatch, next_s_minibatch, terminals_minibatch, self.epoch_finished


In [36]:
rng = np.random.RandomState(2022)

In [37]:
train_val_loader = RLDataLoader(
                    data_dir, rng, 64, 
                    dataset='train_val', pos_samples_in_minibatch=0, 
                    neg_samples_in_minibatch=2)

In [38]:
train_val_loader.make_transition_data(release=True)

DataLoader: making transitions (s,a,r,s') from encoded data structures


# Dataset statistics

Let's look into high-level statistics of each feature (for the appendix)

In [13]:
data = pd.read_csv(os.path.join(data_dir2, "final_trajs_noFill_RAW.csv"), compression="gzip")
data_filled = pd.read_csv(os.path.join(data_dir2, "final_trajs_RAW.csv"), compression="gzip")

In [14]:
data.head()

,m:traj,m:timestep,m:bloc,m:stay_id,m:time_from_sepsis_h,o:gender,o:ventilation,o:admission_age,o:height,o:weight,...,o:spo2,o:bun,o:creatinine,o:inr,o:bilirubin,o:alt,o:ast,o:UO,a:action,r:reward
0,1,0,-8,30004823.0,-8.000000,1,0.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,250.0,0.0,1.0
1,1,1,-7,30004823.0,-7.410000,1,1.0,53.0,175.0,91.0,...,97.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
2,1,2,-6,30004823.0,-6.491667,1,1.0,53.0,175.0,91.0,...,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
3,1,3,-5,30004823.0,-5.491667,1,1.0,53.0,175.0,91.0,...,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0
4,1,4,-4,30004823.0,-4.433333,1,1.0,53.0,175.0,91.0,...,95.0,29.0,0.6,2.3,NaN,NaN,NaN,NaN,15.0,1.0


In [21]:
data_filled.head()

,m:traj,m:timestep,m:bloc,m:stay_id,m:time_from_sepsis_h,o:gender,o:ventilation,o:admission_age,o:weight,o:heart_rate,...,o:spo2,o:bun,o:creatinine,o:inr,o:bilirubin,o:alt,o:ast,o:UO,a:action,r:reward
0,1,0,-8,30004823.0,-8.000000,1,0.0,53.0,91.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,250.0,0.0,1.0
1,1,1,-7,30004823.0,-7.410000,1,1.0,53.0,91.0,88.5,...,97.5,NaN,NaN,NaN,NaN,NaN,NaN,250.0,15.0,1.0
2,1,2,-6,30004823.0,-6.491667,1,1.0,53.0,91.0,86.0,...,94.0,NaN,NaN,NaN,NaN,NaN,NaN,250.0,15.0,1.0
3,1,3,-5,30004823.0,-5.491667,1,1.0,53.0,91.0,87.0,...,95.0,NaN,NaN,NaN,NaN,NaN,NaN,250.0,15.0,1.0
4,1,4,-4,30004823.0,-4.433333,1,1.0,53.0,91.0,95.0,...,95.0,29.0,0.6,2.3,NaN,NaN,NaN,250.0,15.0,1.0


In [15]:
data['m:stay_id'].nunique()

7741

In [16]:
sep_ids = np.load(os.path.join(data_dir2, "sep_subcohort.npy"), allow_pickle=True)

In [18]:
len(sep_ids)

6208

In [19]:
sep_data = data[data['m:stay_id'].isin(sep_ids)].copy()

In [20]:
sep_data['m:stay_id'].nunique()

6188

In [28]:
summary = sep_data.describe()

In [29]:
summary.columns

Index(['m:traj', 'm:timestep', 'm:bloc', 'm:stay_id', 'm:time_from_sepsis_h',
       'o:gender', 'o:ventilation', 'o:admission_age', 'o:height', 'o:weight',
       'o:heart_rate', 'o:sbp', 'o:dbp', 'o:mbp', 'o:resp_rate',
       'o:temperature', 'o:glucose', 'o:so2', 'o:po2', 'o:pco2', 'o:fio2',
       'o:pao2fio2ratio', 'o:ph', 'o:baseexcess', 'o:chloride', 'o:calcium',
       'o:potassium', 'o:sodium', 'o:lactate', 'o:hematocrit', 'o:hemoglobin',
       'o:platelet', 'o:wbc', 'o:albumin', 'o:aniongap', 'o:bicarbonate',
       'o:pt', 'o:ptt', 'o:gcs', 'o:spo2', 'o:bun', 'o:creatinine', 'o:inr',
       'o:bilirubin', 'o:alt', 'o:ast', 'o:UO', 'a:action', 'r:reward'],
      dtype='object')

In [112]:
column = 'o:bilirubin'
summary[column]['50%'], summary[column]['25%'], summary[column]['75%']

(0.9, 0.5, 2.5)

In [126]:
idx = (sep_data['o:admission_age'] >=90)
sep_data[idx]['m:stay_id'].nunique(), sep_data[idx]['m:stay_id'].nunique()/sep_data['m:stay_id'].nunique()

(458, 0.0740142210730446)

In [114]:
sep_data.groupby('m:stay_id').agg({'o:ventilation': 'max'}).mean()

o:ventilation    0.355688
dtype: float64

In [116]:
sep_data.groupby('m:stay_id').agg({'o:ventilation': 'max'}).sum()

o:ventilation    2201.0
dtype: float64

In [117]:
2201 / sep_data['m:stay_id'].nunique()

0.355688429217841

In [131]:
sep_data['fluid_admin'] = (sep_data['a:action']>=5).astype(int)
sep_data['vaso_admin'] = (sep_data['a:action']%5 != 0).astype(int)

In [132]:
sep_data.head()

,m:traj,m:timestep,m:bloc,m:stay_id,m:time_from_sepsis_h,o:gender,o:ventilation,o:admission_age,o:height,o:weight,...,o:creatinine,o:inr,o:bilirubin,o:alt,o:ast,o:UO,a:action,r:reward,fluid_admin,vaso_admin
0,1,0,-8,30004823.0,-8.000000,1,0.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,250.0,0.0,1.0,0,0
1,1,1,-7,30004823.0,-7.410000,1,1.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0,1,0
2,1,2,-6,30004823.0,-6.491667,1,1.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0,1,0
3,1,3,-5,30004823.0,-5.491667,1,1.0,53.0,175.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,15.0,1.0,1,0
4,1,4,-4,30004823.0,-4.433333,1,1.0,53.0,175.0,91.0,...,0.6,2.3,NaN,NaN,NaN,NaN,15.0,1.0,1,0


In [135]:
sep_data.groupby('m:stay_id').agg({'fluid_admin': 'max'}).sum(), sep_data.groupby('m:stay_id').agg({'fluid_admin': 'max'}).sum() / sep_data['m:stay_id'].nunique()

(fluid_admin    6032
 dtype: int64,
 fluid_admin    0.97479
 dtype: float64)

In [136]:
sep_data.groupby('m:stay_id').agg({'vaso_admin': 'max'}).sum(), sep_data.groupby('m:stay_id').agg({'vaso_admin': 'max'}).sum() / sep_data['m:stay_id'].nunique()

(vaso_admin    2241
 dtype: int64,
 vaso_admin    0.362153
 dtype: float64)